# 01. Consolidated Data Preprocessing

## Objective
This notebook cleans and transforms the consolidated project-level dataset used for monitoring solar system performance.

## Input
- Raw consolidated dataset (`consolidated_bronze.csv`)

## Output
- Cleaned analytical dataset for downstream analysis and dashboarding.

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import re
import glob
import os



## Data Loading
This section loads the consolidated dataset used for analysis.

In [71]:
file_path = "../data/bronze/consolidated_bronze.csv"

df = pd.read_csv(file_path, sep=";", encoding="utf-8")

df.head()

,id_proyecto,fecha,rendimiento fv (kwh),energia acumulada (kwh),energia exportada (kwh),importacion (kwh),energia especifica (kwh/kwp),consumo (kwh),autoconsumo (kwh),potencia maxima (kw),...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,EVT-2021-0001,01/01/2025,151.34,0.00,0.0,0.0,25.712,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EVT-2021-0002,01/01/2025,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EVT-2021-0003,01/01/2025,77.80,0.00,0.0,0.0,27.184,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EVT-2022-0004,01/01/2025,156.80,0.00,0.0,0.0,25.712,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EVT-2022-0005,01/01/2025,6.51,9086.07,0.0,0.0,2.893,0.0,0.0,2.009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Exploration

Initial inspection of dataset structure, data types, and basic statistics.

In [72]:
df.shape

(16934, 21)

In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16934 entries, 0 to 16933
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_proyecto                   16934 non-null  str    
 1   fecha                         16934 non-null  str    
 2   rendimiento fv (kwh)          16934 non-null  float64
 3   energia acumulada (kwh)       16934 non-null  float64
 4   energia exportada (kwh)       16934 non-null  float64
 5   importacion (kwh)             16934 non-null  float64
 6   energia especifica (kwh/kwp)  16934 non-null  str    
 7   consumo (kwh)                 16934 non-null  float64
 8   autoconsumo (kwh)             16934 non-null  float64
 9   potencia maxima (kw)          16934 non-null  str    
 10  co2 evitado (t)               16934 non-null  str    
 11  Unnamed: 11                   7 non-null      str    
 12  Unnamed: 12                   7 non-null      float64
 13  Unnamed: 13 

In [74]:
df.describe()

,rendimiento fv (kwh),energia acumulada (kwh),energia exportada (kwh),importacion (kwh),consumo (kwh),autoconsumo (kwh),Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
count,16934.000000,1.693400e+04,1.693400e+04,1.693400e+04,16934.000000,1.693400e+04,7.000000,7.0,7.0,7.0,7.000000,7.0,7.0,7.0,7.000000
mean,253.606386,5.537309e+04,1.489129e+02,1.034515e+02,165.389377,1.272020e+02,120.090000,0.0,0.0,0.0,20.402571,0.0,0.0,0.0,0.010800
std,998.087909,1.627051e+05,1.201327e+04,1.091074e+04,1083.343804,8.442050e+03,89.119588,0.0,0.0,0.0,15.140887,0.0,0.0,0.0,0.008018
min,0.000000,0.000000e+00,-1.098464e+06,-1.003692e+06,-94773.100000,0.000000e+00,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000
25%,15.500000,1.564645e+03,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,56.265000,0.0,0.0,0.0,9.559000,0.0,0.0,0.0,0.005050
50%,60.050000,1.661208e+04,7.225000e+00,2.327000e+01,47.200000,2.042500e+01,139.880000,0.0,0.0,0.0,23.765000,0.0,0.0,0.0,0.012600
75%,144.090000,4.453900e+04,3.450750e+01,9.670500e+01,173.755000,7.100750e+01,187.040000,0.0,0.0,0.0,31.777000,0.0,0.0,0.0,0.016800
max,36712.000000,1.954029e+06,1.098479e+06,1.003693e+06,94783.100000,1.098475e+06,214.140000,0.0,0.0,0.0,36.381000,0.0,0.0,0.0,0.019300


## Data Cleaning

Handling duplicates and missing values.

In [75]:
df.duplicated().sum()

np.int64(2)

In [76]:
df = df.drop_duplicates()

In [77]:
df.isnull().sum()

id_proyecto                         0
fecha                               0
rendimiento fv (kwh)                0
energia acumulada (kwh)             0
energia exportada (kwh)             0
importacion (kwh)                   0
energia especifica (kwh/kwp)        0
consumo (kwh)                       0
autoconsumo (kwh)                   0
potencia maxima (kw)                0
co2 evitado (t)                     0
Unnamed: 11                     16925
Unnamed: 12                     16925
Unnamed: 13                     16925
Unnamed: 14                     16925
Unnamed: 15                     16925
Unnamed: 16                     16925
Unnamed: 17                     16925
Unnamed: 18                     16925
Unnamed: 19                     16925
Unnamed: 20                     16925
dtype: int64

In [78]:
(df.isnull().sum() / len(df)) * 100

id_proyecto                      0.000000
fecha                            0.000000
rendimiento fv (kwh)             0.000000
energia acumulada (kwh)          0.000000
energia exportada (kwh)          0.000000
importacion (kwh)                0.000000
energia especifica (kwh/kwp)     0.000000
consumo (kwh)                    0.000000
autoconsumo (kwh)                0.000000
potencia maxima (kw)             0.000000
co2 evitado (t)                  0.000000
Unnamed: 11                     99.958658
Unnamed: 12                     99.958658
Unnamed: 13                     99.958658
Unnamed: 14                     99.958658
Unnamed: 15                     99.958658
Unnamed: 16                     99.958658
Unnamed: 17                     99.958658
Unnamed: 18                     99.958658
Unnamed: 19                     99.958658
Unnamed: 20                     99.958658
dtype: float64

In [79]:
cols_to_drop = [col for col in df.columns if "Unnamed" in col]
df = df.drop(columns=cols_to_drop)

df.head()

,id_proyecto,fecha,rendimiento fv (kwh),energia acumulada (kwh),energia exportada (kwh),importacion (kwh),energia especifica (kwh/kwp),consumo (kwh),autoconsumo (kwh),potencia maxima (kw),co2 evitado (t)
0,EVT-2021-0001,01/01/2025,151.34,0.00,0.0,0.0,25.712,0.0,0.0,0.0,0.0136
1,EVT-2021-0002,01/01/2025,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,EVT-2021-0003,01/01/2025,77.80,0.00,0.0,0.0,27.184,0.0,0.0,0.0,0.007
3,EVT-2022-0004,01/01/2025,156.80,0.00,0.0,0.0,25.712,0.0,0.0,0.0,0.013
4,EVT-2022-0005,01/01/2025,6.51,9086.07,0.0,0.0,2.893,0.0,0.0,2.009,0.003


Columns with extremely high missing values (Unnamed columns) were removed, as they do not provide useful information for analysis.

In [80]:
# Convert date column
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors="coerce")

# Convert numeric columns (handling comma decimals)
numeric_cols = [
    "rendimiento fv (kwh)",
    "energia acumulada (kwh)",
    "energia exportada (kwh)",
    "importacion (kwh)",
    "energia especifica (kwh/kwp)",
    "consumo (kwh)",
    "autoconsumo (kwh)",
    "potencia maxima (kw)",
    "co2 evitado (t)"
]

for col in numeric_cols:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.replace(",", ".", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

Date and numeric columns were standardized to ensure consistency and enable accurate time-based and quantitative analysis.

## Monthly Generation Calculation

Monthly energy generation was calculated by aggregating daily energy production (`rendimiento fv (kwh)`) for each project.

This metric provides a clearer view of system performance over time and enables trend analysis, anomaly detection, and comparison across different periods.

In [81]:
# Create time features
df["anio"] = df["fecha"].dt.year
df["mes_num"] = df["fecha"].dt.month
df["month_year"] = df["fecha"].dt.to_period("M")

# Monthly generation by project
monthly_generation = (
    df.groupby(["id_proyecto", "month_year"], as_index=False)["rendimiento fv (kwh)"]
    .sum()
    .rename(columns={"rendimiento fv (kwh)": "generacion_mensual_kwh"})
)

# Merge back to original dataframe
df = df.merge(monthly_generation, on=["id_proyecto", "month_year"], how="left")

df.head()

,id_proyecto,fecha,rendimiento fv (kwh),energia acumulada (kwh),energia exportada (kwh),importacion (kwh),energia especifica (kwh/kwp),consumo (kwh),autoconsumo (kwh),potencia maxima (kw),co2 evitado (t),anio,mes_num,month_year,generacion_mensual_kwh
0,EVT-2021-0001,2025-01-01,151.34,0.00,0.0,0.0,25.712,0.0,0.0,0.000,0.0136,2025,1,2025-01,5236.83
1,EVT-2021-0002,2025-01-01,0.00,0.00,0.0,0.0,0.000,0.0,0.0,0.000,0.0000,2025,1,2025-01,2534.99
2,EVT-2021-0003,2025-01-01,77.80,0.00,0.0,0.0,27.184,0.0,0.0,0.000,0.0070,2025,1,2025-01,195.74
3,EVT-2022-0004,2025-01-01,156.80,0.00,0.0,0.0,25.712,0.0,0.0,0.000,0.0130,2025,1,2025-01,5577.70
4,EVT-2022-0005,2025-01-01,6.51,9086.07,0.0,0.0,2.893,0.0,0.0,2.009,0.0030,2025,1,2025-01,285.93


In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16932 entries, 0 to 16931
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id_proyecto                   16932 non-null  str           
 1   fecha                         16932 non-null  datetime64[us]
 2   rendimiento fv (kwh)          16932 non-null  float64       
 3   energia acumulada (kwh)       16932 non-null  float64       
 4   energia exportada (kwh)       16932 non-null  float64       
 5   importacion (kwh)             16932 non-null  float64       
 6   energia especifica (kwh/kwp)  14021 non-null  float64       
 7   consumo (kwh)                 16932 non-null  float64       
 8   autoconsumo (kwh)             16932 non-null  float64       
 9   potencia maxima (kw)          15311 non-null  float64       
 10  co2 evitado (t)               16923 non-null  float64       
 11  anio                          16932 non

In [83]:
df.to_csv("../data/silver/consolidated_silver.csv", index=False)